# Canonical GHZ3 Bell baseline

This notebook prepares a deterministic `canonical_ez` direct-basis input for the three-qutrit GHZ Bell experiment. By default, Run All submits only the local Aer baseline; IQM submits a remote job only after `RUN_IQM` is set to `True`. Credentials remain provider/environment-only and are never embedded or persisted by this notebook.

In [ ]:
import hashlib
import json
import sys
from pathlib import Path
from uuid import uuid4

import numpy as np
import qiskit.qpy as qpy
from qiskit.quantum_info import Statevector


def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "qudits_on_qubits"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "Cannot find repository root. Start from this repository or a descendant "
        "containing pyproject.toml and src/qudits_on_qubits."
    )


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qudits_on_qubits.reference_experiments import get_encoding, get_reference_experiment
from qudits_on_qubits.benchmarks.direct_basis.circuits import build_direct_basis_graph_state_circuit
from qudits_on_qubits.experiments import (
    AerIdeal,
    BootstrapConfig,
    ExperimentSpec,
    IQMHardware,
    MitigationConfig,
    PathBasis,
    run_experiment,
)

In [ ]:
EXPECTED_STATE = "ghz3"
EXPECTED_QUBITS = 6


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def load_single_circuit(path):
    try:
        with Path(path).open("rb") as handle:
            circuits = qpy.load(handle)
    except Exception as error:
        raise RuntimeError(f"unable to load canonical basis QPY: {path}") from error
    if len(circuits) != 1:
        raise RuntimeError(
            f"canonical basis QPY must contain exactly one circuit, found {len(circuits)}"
        )
    return circuits[0]


def validate_canonical_basis(directory, expected_encoding, expected_circuit):
    directory = Path(directory)
    required_files = {"graph_state_direct_basis.qpy", "E.npy", "metadata.json"}
    try:
        actual_files = {path.name for path in directory.iterdir()}
    except OSError as error:
        raise RuntimeError(
            f"canonical basis directory is unavailable: {directory}"
        ) from error
    if actual_files != required_files:
        raise RuntimeError(
            "canonical basis files must be exactly graph_state_direct_basis.qpy, E.npy, "
            f"and metadata.json; found {sorted(actual_files)}"
        )

    encoding_path = directory / "E.npy"
    try:
        encoding = np.load(encoding_path, allow_pickle=False)
    except Exception as error:
        raise RuntimeError(
            f"canonical basis encoding is invalid: {encoding_path}"
        ) from error
    if encoding.shape != (4, 3):
        raise RuntimeError(
            f"canonical basis encoding must have shape (4, 3), got {encoding.shape}"
        )
    try:
        is_finite = np.isfinite(encoding).all()
    except TypeError as error:
        raise RuntimeError(
            "canonical basis encoding must be numeric and finite"
        ) from error
    if not is_finite:
        raise RuntimeError("canonical basis encoding must be finite")
    if not np.allclose(
        encoding.conj().T @ encoding, np.eye(3), atol=1e-12, rtol=0
    ):
        raise RuntimeError("canonical basis encoding must be an isometry")
    if not np.array_equal(encoding, expected_encoding):
        raise RuntimeError(
            "canonical basis encoding does not match canonical_ez"
        )

    circuit_path = directory / "graph_state_direct_basis.qpy"
    circuit = load_single_circuit(circuit_path)
    if circuit.num_qubits != EXPECTED_QUBITS or circuit.num_clbits != 0:
        raise RuntimeError(
            "canonical basis QPY must contain one unmeasured six-qubit circuit"
        )
    for instruction in circuit.data:
        operation = instruction.operation
        if operation.name in {"measure", "reset"}:
            raise RuntimeError(
                "canonical basis QPY must not contain measurements or resets"
            )
        if getattr(operation, "condition", None) is not None:
            raise RuntimeError(
                "canonical basis QPY must not contain conditioned instructions"
            )
        if getattr(operation, "blocks", ()):
            raise RuntimeError(
                "canonical basis QPY must not contain control flow"
            )
    try:
        same_state = Statevector.from_instruction(circuit).equiv(
            Statevector.from_instruction(expected_circuit)
        )
    except Exception as error:
        raise RuntimeError(
            "canonical basis QPY circuit cannot be validated as a state preparation"
        ) from error
    if not same_state:
        raise RuntimeError(
            "canonical basis QPY circuit does not match the canonical graph state"
        )

    metadata_path = directory / "metadata.json"
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    except Exception as error:
        raise RuntimeError(
            f"canonical basis metadata is invalid: {metadata_path}"
        ) from error
    expected_metadata = {
        "schema": "qoq-reference-basis-v1",
        "state": EXPECTED_STATE,
        "encoding_id": "canonical_ez",
        "num_qubits": EXPECTED_QUBITS,
        "encoding_shape": [4, 3],
        "files": {
            "graph_state_direct_basis.qpy": {
                "sha256": sha256_file(circuit_path)
            },
            "E.npy": {"sha256": sha256_file(encoding_path)},
        },
    }
    if metadata != expected_metadata:
        raise RuntimeError(
            "canonical basis metadata does not match the validated bundle"
        )


def prepare_canonical_basis(repo_root):
    repo_root = Path(repo_root)
    expected_encoding = get_encoding("canonical_ez").as_array()
    expected_circuit = build_direct_basis_graph_state_circuit(
        EXPECTED_STATE, expected_encoding
    )
    parent = (
        repo_root / "experiment_inputs" / "reference_bases" / EXPECTED_STATE
    )
    directory = parent / "canonical_ez"

    if directory.exists():
        validate_canonical_basis(directory, expected_encoding, expected_circuit)
        return directory

    parent.mkdir(parents=True, exist_ok=True)
    staging_directory = parent / f".canonical_ez.tmp-{uuid4().hex}"
    staging_directory.mkdir()
    staging_files = (
        staging_directory / "graph_state_direct_basis.qpy",
        staging_directory / "E.npy",
        staging_directory / "metadata.json",
    )

    def cleanup_staging():
        for staging_file in staging_files:
            if staging_file.exists():
                staging_file.unlink()
        if staging_directory.exists():
            staging_directory.rmdir()

    try:
        qpy_path, encoding_path, metadata_path = staging_files
        with qpy_path.open("wb") as handle:
            qpy.dump(expected_circuit, handle)
        with encoding_path.open("wb") as handle:
            np.save(handle, expected_encoding, allow_pickle=False)
        metadata = {
            "schema": "qoq-reference-basis-v1",
            "state": EXPECTED_STATE,
            "encoding_id": "canonical_ez",
            "num_qubits": EXPECTED_QUBITS,
            "encoding_shape": [4, 3],
            "files": {
                "graph_state_direct_basis.qpy": {
                    "sha256": sha256_file(qpy_path)
                },
                "E.npy": {"sha256": sha256_file(encoding_path)},
            },
        }
        metadata_path.write_text(
            json.dumps(metadata, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )

        validate_canonical_basis(
            staging_directory, expected_encoding, expected_circuit
        )
        try:
            staging_directory.rename(directory)
        except FileExistsError:
            cleanup_staging()
            validate_canonical_basis(
                directory, expected_encoding, expected_circuit
            )
        return directory
    finally:
        cleanup_staging()

In [ ]:
CANONICAL_BASIS_DIRECTORY = prepare_canonical_basis(REPO_ROOT)
CANONICAL_BASIS_DIRECTORY

## Shared configuration

The canonical reference, uncertainty settings, and hardware mitigation policy are shared across the Aer and IQM baselines.

In [ ]:
SHOTS = 100
UNCERTAINTY = BootstrapConfig(samples=2_000, seed=7)
HARDWARE_MITIGATION = MitigationConfig(
    readout=True,
    zne=True,
    zne_factors=(1, 3, 5),
)
REFERENCE = get_reference_experiment("ghz3")
RESULTS = {}

## Aer ideal baseline

This unguarded local run records the canonical ideal-backend result.

In [ ]:
AER_SPEC = ExperimentSpec(
    state="ghz3",
    basis=PathBasis(CANONICAL_BASIS_DIRECTORY),
    backend=AerIdeal(seed_simulator=11),
    shots=SHOTS,
    uncertainty=UNCERTAINTY,
    tags={"baseline": "canonical_ez", "backend": "aer_ideal"},
)
AER_RESULT = run_experiment(AER_SPEC, repo_root=REPO_ROOT)
RESULTS["aer_ideal"] = AER_RESULT

## IQM Garnet baseline

Submission is opt-in; the default keeps this hardware run skipped.

In [ ]:
RUN_IQM = False

if RUN_IQM:
    IQM_SPEC = ExperimentSpec(
        state="ghz3",
        basis=PathBasis(CANONICAL_BASIS_DIRECTORY),
        backend=IQMHardware(device="garnet", use_metrics=True),
        shots=SHOTS,
        mitigation=HARDWARE_MITIGATION,
        uncertainty=UNCERTAINTY,
        tags={"baseline": "canonical_ez", "backend": "iqm_garnet"},
    )
    IQM_RESULT = run_experiment(IQM_SPEC, repo_root=REPO_ROOT)
    RESULTS["iqm_garnet"] = IQM_RESULT
else:
    print("IQM Garnet skipped; set RUN_IQM = True to submit.")

## Result summary

The summary reuses runner-produced values without recomputation and includes the frozen reference bounds.

In [ ]:
def summarize_results(results, reference):
    rows = []
    for backend, missing_status in (
        ("aer_ideal", "not_run"),
        ("iqm_garnet", "skipped"),
    ):
        result = results.get(backend)
        rows.append(
            {
                "backend": backend,
                "status": (
                    missing_status if result is None else result.status.value
                ),
                "raw": None if result is None else result.values.get("raw"),
                "readout_mitigated": (
                    None
                    if result is None
                    else result.values.get("readout_mitigated")
                ),
                "zne": None if result is None else result.values.get("zne"),
                "zne_readout_mitigated": (
                    None
                    if result is None
                    else result.values.get("zne_readout_mitigated")
                ),
                "diagnostics": (
                    None if result is None else result.values.get("diagnostics")
                ),
                "leakage_rate": (
                    None if result is None else result.values.get("leakage_rate")
                ),
                "classical_bound": reference.bell_functional.classical_bound,
                "ideal_bell_value": reference.expected.ideal_bell_value,
                "artifact_dir": (
                    None if result is None else str(result.artifact_dir)
                ),
            }
        )
    return rows

In [ ]:
SUMMARY = summarize_results(RESULTS, REFERENCE)
SUMMARY